# Single User Resource Allocation Model

This notebook is the reviewer-facing walkthrough of the shared single-user engine. It shows one user,
one distance, and one rate requirement, then exposes the candidate enumeration and feasibility chain
without doing any multi-distance sweep logic.


## 1. Explicit Inputs


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append(str((Path.cwd() / "src").resolve()))

from downlink_resource_allocation.pa_models import build_pa_catalog, build_pa_characteristics_table
from downlink_resource_allocation.single_user_engine import SingleUserResourceAllocationEngine


In [ ]:
# Shared baseline assumptions for the single-user model.
# Downstream notebooks can override these dictionaries after loading this notebook.

# speed of light in m/s
C_MPS = 299_792_458.0

link_constants = {
    "pl_model": "umi_sc_nlos",  # umi_sc_nlos / fspl / umi_sc_los
    "fc_hz": 3.5e9,
    "g_tx_db": 8.0,
    "g_rx_db": 0.0,
    "n0_dbm_per_hz": -174.0,
    "lna_noise_figure_db": 5.0,
    "shadow_margin_db": 4.0,
}

phy_constants = {
    "channel_bw_hz": 100e6,
    "l_impl_db": 3.0,
    "mi_n_samples": 1500,
    "papr_db": 8.0,
    "g_phi": 1.0,
    "sigma_phi2": 0.0,
    "sigma_q2": 0.0,
    "n_dmrs_sym": 2,
    "n_sym_data": 12,
    "n_sym_total": 14,
    "dft_size_N": 4096,
    "n_slots_win": 7,
    "t_slot_s": 0.5e-3,
    "n_tx_chains": 4,
    "use_psd_constraint": True,
    "psd_max_w_per_hz": 8e-7,
    "delta_f_hz": 30e3,
}

scheduler_sweep = {
    "bandwidth_space_hz": (100e6, 50e6),
    "layers_space": [1, 2, 4],
    "mcs_space": list(range(0, 29)),
    "prb_step": 5,
}

# 3GPP-like MCS lookup used by the link abstraction.
mcs_table = {
    0: {"qm": 2, "r": 120, "eta": 0.2344},
    1: {"qm": 2, "r": 157, "eta": 0.3066},
    2: {"qm": 2, "r": 193, "eta": 0.3770},
    3: {"qm": 2, "r": 251, "eta": 0.4902},
    4: {"qm": 2, "r": 308, "eta": 0.6016},
    5: {"qm": 2, "r": 379, "eta": 0.7402},
    6: {"qm": 2, "r": 449, "eta": 0.8770},
    7: {"qm": 2, "r": 526, "eta": 1.0273},
    8: {"qm": 2, "r": 602, "eta": 1.1758},
    9: {"qm": 2, "r": 679, "eta": 1.3262},
    10: {"qm": 4, "r": 340, "eta": 1.3281},
    11: {"qm": 4, "r": 378, "eta": 1.4766},
    12: {"qm": 4, "r": 434, "eta": 1.6953},
    13: {"qm": 4, "r": 490, "eta": 1.9141},
    14: {"qm": 4, "r": 553, "eta": 2.1602},
    15: {"qm": 4, "r": 616, "eta": 2.4063},
    16: {"qm": 4, "r": 658, "eta": 2.5703},
    17: {"qm": 6, "r": 438, "eta": 2.5664},
    18: {"qm": 6, "r": 466, "eta": 2.7305},
    19: {"qm": 6, "r": 517, "eta": 3.0293},
    20: {"qm": 6, "r": 567, "eta": 3.3223},
    21: {"qm": 6, "r": 616, "eta": 3.6094},
    22: {"qm": 6, "r": 666, "eta": 3.9023},
    23: {"qm": 6, "r": 719, "eta": 4.2129},
    24: {"qm": 6, "r": 772, "eta": 4.5234},
    25: {"qm": 6, "r": 822, "eta": 4.8164},
    26: {"qm": 6, "r": 873, "eta": 5.1152},
    27: {"qm": 6, "r": 910, "eta": 5.3320},
    28: {"qm": 6, "r": 948, "eta": 5.5547},
}

PA_DATA_CSV = os.path.join("PA models", "3.5Ghz_pas", "4W_8W_NR_combined_NR_carrier.csv")



user_case = {
    "distance_m": 100.0,
    "rate_target_bps": 80e6,
}

link_constants_table = pd.DataFrame([link_constants])
phy_constants_table = pd.DataFrame([phy_constants])
scheduler_sweep_table = pd.DataFrame(
    [
        {
            "bandwidth_space_hz": str(tuple(float(v) for v in scheduler_sweep["bandwidth_space_hz"])),
            "layers_space": str(list(scheduler_sweep["layers_space"])),
            "mcs_space": f"{min(scheduler_sweep['mcs_space'])}..{max(scheduler_sweep['mcs_space'])}",
            "prb_step": int(scheduler_sweep["prb_step"]),
        }
    ]
)
user_case_table = pd.DataFrame([user_case])

display(link_constants_table)
display(phy_constants_table)
display(scheduler_sweep_table)
display(user_case_table)



user_case = {
    "distance_m": 100.0,
    "rate_target_bps": 80e6,
}

link_constants_table = pd.DataFrame([link_constants])
phy_constants_table = pd.DataFrame([phy_constants])
scheduler_sweep_table = pd.DataFrame(
    [
        {
            "bandwidth_space_hz": str(tuple(float(v) for v in scheduler_sweep["bandwidth_space_hz"])),
            "layers_space": str(list(scheduler_sweep["layers_space"])),
            "mcs_space": f"{min(scheduler_sweep['mcs_space'])}..{max(scheduler_sweep['mcs_space'])}",
            "prb_step": int(scheduler_sweep["prb_step"]),
        }
    ]
)
user_case_table = pd.DataFrame([user_case])

display(link_constants_table)
display(phy_constants_table)
display(scheduler_sweep_table)
display(user_case_table)



user_case = {
    "distance_m": 100.0,
    "rate_target_bps": 80e6,
}

link_constants_table = pd.DataFrame([link_constants])
phy_constants_table = pd.DataFrame([phy_constants])
scheduler_sweep_table = pd.DataFrame(
    [
        {
            "bandwidth_space_hz": str(tuple(float(v) for v in scheduler_sweep["bandwidth_space_hz"])),
            "layers_space": str(list(scheduler_sweep["layers_space"])),
            "mcs_space": f"{min(scheduler_sweep['mcs_space'])}..{max(scheduler_sweep['mcs_space'])}",
            "prb_step": int(scheduler_sweep["prb_step"]),
        }
    ]
)
user_case_table = pd.DataFrame([user_case])

display(link_constants_table)
display(phy_constants_table)
display(scheduler_sweep_table)
display(user_case_table)



user_case = {
    "distance_m": 100.0,
    "rate_target_bps": 80e6,
}

link_constants_table = pd.DataFrame([link_constants])
phy_constants_table = pd.DataFrame([phy_constants])
scheduler_sweep_table = pd.DataFrame(
    [
        {
            "bandwidth_space_hz": str(tuple(float(v) for v in scheduler_sweep["bandwidth_space_hz"])),
            "layers_space": str(list(scheduler_sweep["layers_space"])),
            "mcs_space": f"{min(scheduler_sweep['mcs_space'])}..{max(scheduler_sweep['mcs_space'])}",
            "prb_step": int(scheduler_sweep["prb_step"]),
        }
    ]
)
user_case_table = pd.DataFrame([user_case])

display(link_constants_table)
display(phy_constants_table)
display(scheduler_sweep_table)
display(user_case_table)


## 2. Shared Engine


In [ ]:
pa_input_table = pd.read_csv(PA_DATA_CSV)
pa_catalog = build_pa_catalog(PA_DATA_CSV)
pa_characteristics = build_pa_characteristics_table(pa_catalog)

engine = SingleUserResourceAllocationEngine(
    link_constants=link_constants,
    phy_constants=phy_constants,
    scheduler_sweep=scheduler_sweep,
    mcs_table=mcs_table,
    pa_catalog=pa_catalog,
)

print(f"PA input rows: {len(pa_input_table):,}")
print(f"PA models in catalog: {len(pa_catalog)}")
display(pa_characteristics)


## 3. Problem Construction


In [ ]:
problem = engine.build_single_user_problem(distance_m=user_case["distance_m"])
problem_views = engine.describe_problem(problem)
search_space_summary = pd.DataFrame([engine.estimate_search_space(problem, [user_case])])

display(problem_views["deployment_summary"])
display(problem_views["rrc_catalog"])
display(problem_views["search_space_summary"])
display(search_space_summary)


## 4. Candidate Enumeration And Evaluation


In [ ]:
PUBLIC_CONFIG_COLUMNS = [
    "distance_m",
    "path_loss_db",
    "pa_id",
    "pa_name",
    "rate_ach_bps",
    "p_dc_avg_total_w",
    "layers",
    "mcs",
    "n_prb",
    "n_slots_on",
    "alpha_f",
    "alpha_t",
    "bandwidth_hz",
    "n_active_tx",
    "p_out_total_w",
    "ps_total_w",
    "gamma_req_lin",
]

candidate_preview = pd.DataFrame(
    [candidate.__dict__ for _, candidate in zip(range(10), engine.enumerate_scheduler_candidates(problem))]
)
candidate_table = engine.evaluate_candidate_table(
    problem,
    include_infeasible=True,
    required_rate_bps=user_case["rate_target_bps"],
)
feasible_table = engine.filter_feasible_configurations(
    candidate_table,
    required_rate_bps=user_case["rate_target_bps"],
)

feasibility_summary = (
    candidate_table.groupby(["is_feasible", "infeasibility_reason"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["is_feasible", "count"], ascending=[False, False])
    .reset_index(drop=True)
)

best_feasible = feasible_table[PUBLIC_CONFIG_COLUMNS].sort_values(
    ["p_dc_avg_total_w", "bandwidth_hz", "n_prb", "n_slots_on"]
).reset_index(drop=True)

display(candidate_preview)
display(feasibility_summary)
display(best_feasible.head(12))


## 5. Compact Feasible-Space Readout


In [ ]:
feasible_summary_by_pa = (
    feasible_table.groupby(["pa_id", "pa_name"], dropna=False)
    .agg(
        feasible_rows=("rate_ach_bps", "size"),
        min_rate_bps=("rate_ach_bps", "min"),
        max_rate_bps=("rate_ach_bps", "max"),
        min_power_w=("p_dc_avg_total_w", "min"),
        max_power_w=("p_dc_avg_total_w", "max"),
    )
    .reset_index()
)
display(feasible_summary_by_pa)

fig, ax = plt.subplots(figsize=(8, 5))
for pa_id, df_pa in feasible_table.groupby("pa_id", sort=True):
    ax.scatter(
        df_pa["rate_ach_bps"] / 1e6,
        df_pa["p_dc_avg_total_w"],
        s=18,
        alpha=0.7,
        label=df_pa["pa_name"].iloc[0],
    )
ax.set_xlabel("Achievable rate (Mbps)")
ax.set_ylabel("Window-averaged total PA DC power (W)")
ax.set_title("Feasible single-user operating points")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
